In [131]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from astroquery.gaia import Gaia
import matplotlib as mpl
mpl.rcdefaults()  
plt.rcParams['text.usetex'] = True
from dustmaps.bayestar import BayestarQuery
from astropy.coordinates import SkyCoord
import astropy.units as u
bayestar = BayestarQuery(version='bayestar2019')

from cmcrameri import cm
from sklearn.linear_model import LinearRegression
from scipy.interpolate import griddata
from matplotlib.ticker import FixedLocator
from matplotlib.lines import Line2D

Loading pixel_info ...
Loading samples ...
Loading best_fit ...
Replacing NaNs in reliable distance estimates ...
Sorting pixel_info ...
Extracting hp_idx_sorted and data_idx at each nside ...
  nside = 64
  nside = 128
  nside = 256
  nside = 512
  nside = 1024
t = 27.573 s
  pix_info:   0.352 s
   samples:  14.707 s
      best:   2.979 s
       nan:   0.032 s
      sort:   9.450 s
       idx:   0.053 s


In [157]:
def numax_to_logg(numax, Teff):
    """Scaling relation: logg from numax at a fixed Teff."""
    return LOGG_SUN + np.log10(numax / NUMAX_SUN) + 0.5 * np.log10(Teff / TEFF_SUN)

def logg_to_numax(logg, Teff):
    """Inverse: numax from logg at a fixed Teff."""
    return NUMAX_SUN * 10**(logg - LOGG_SUN - 0.5 * np.log10(Teff / TEFF_SUN))

# Solar reference values
NUMAX_SUN = 3090.0  # μHz
TEFF_SUN = 5777.0   # K
LOGG_SUN = 4.44     
G_SUN = 27420
M_SUN = 1.989*10**30  #kg
DNU_SUN = 135.1  # μHz
LOGG_SUN  = 4.438

In [158]:
ms = pd.read_csv('../data/true_mult_solution.csv',dtype={'source_id': str})
sed_results = pd.read_csv('../data/sed_results.csv',dtype={'source_id': str})

In [160]:
# Source IDs available in sed_results
valid_ids = set(sed_results['source_id'])

# Keep a fname only if every associated source_id is present
keep = (
    ms.groupby('fname')['source_id']
      .transform(lambda ids: ids.isin(valid_ids).all())
)

ms = ms.loc[keep]

In [161]:
ms

,fname,source_id,numax,numax_sig,dnu,dnu_sig,certainty,syd_success,KIC,GaiaDR3
10,gaia-dr3-2073541323539522304,2073541323552464512,20.40812,1.14177,2.72748,0.06131,1,2.0,NaN,NaN
11,gaia-dr3-2073541323539522304,2073541319242386176,20.40812,1.14177,2.72748,0.06131,1,2.0,NaN,NaN
12,gaia-dr3-2073562798390143616,2073562798389496320,34.32275,1.68715,4.12786,0.04718,1,2.0,NaN,NaN
13,gaia-dr3-2073562798390143616,2073562798390143616,34.32275,1.68715,4.12786,0.04718,1,2.0,NaN,NaN
25,gaia-dr3-2073776344155148800,2073776344155147520,49.44037,3.23882,4.91286,0.09938,1,1.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
786,kic-011338438,2131992353798606208,15.57669,0.82501,2.14227,0.01828,1,2.0,11338437.0,2.131992e+18
787,kic-011361661,2086825687444210176,51.98220,3.05548,5.26299,0.09671,1,1.0,NaN,NaN
788,kic-011361661,2086825691742381312,51.98220,3.05548,5.26299,0.09671,1,1.0,11361659.0,2.086826e+18
798,kic-005906704,2075121046883575680,62.06649,2.30365,6.25654,0.03253,1,2.0,NaN,NaN


In [162]:
source_ids = ms['source_id']
batch_size=1000
all_results = []
for i in range(0, len(source_ids), batch_size):
    batch = source_ids[i:i + batch_size]  
    id_string = ','.join(map(str, batch))
        
    query = f"""
        SELECT source_id, ra, dec, phot_g_mean_mag, phot_g_mean_flux, phot_g_mean_flux_error, parallax, parallax_error
        FROM gaiadr3.gaia_source
        WHERE source_id IN ({id_string})
        """
        
    job = Gaia.launch_job(query)
    all_results.append(job.get_results().to_pandas())
all_results = pd.concat(all_results)
all_results['source_id'] = all_results['source_id'].astype(str)

In [163]:
good_parallax = all_results[all_results['parallax'] / all_results['parallax_error'] >= 1]
parallax_problem = all_results[~all_results['source_id'].isin(good_parallax['source_id'])]

In [164]:
#deal with stars with good parallax first
# distance from parallax
good_parallax['distance_pc'] = 1000 / good_parallax['parallax']

coords = SkyCoord(
    ra=good_parallax['ra'].values * u.deg,
    dec=good_parallax['dec'].values * u.deg,
    distance=good_parallax['distance_pc'].values * u.pc
)

# BAYESTAR returns reddening on its own scale — need mode='best' or samples
good_parallax['E_bayestar'] = bayestar(coords, mode='best')

E_BV = 0.884 * good_parallax['E_bayestar']
#convert to reddening in G band
good_parallax['A_G'] =  2.74* E_BV

# extinction-corrected apparent magnitude
good_parallax['G_corrected'] = good_parallax['phot_g_mean_mag'] - good_parallax['A_G']

# now redo the earlier steps with G_corrected instead of phot_g_mean_mag
good_parallax['abs_mag'] = good_parallax['G_corrected'] + 5 * np.log10(good_parallax['parallax']) - 10

# --- error propagation ---
# magnitude error from flux error: sigma_mag = (2.5 / ln10) * (flux_error / flux)
good_parallax['mag_error'] = (2.5 / np.log(10)) * (good_parallax['phot_g_mean_flux_error'] / good_parallax['phot_g_mean_flux'])

# parallax contributes to abs_mag error via d(abs_mag)/d(parallax) = 5/(parallax * ln10)
good_parallax['parallax_mag_error'] = (5 / np.log(10)) * (good_parallax['parallax_error'] / good_parallax['parallax'])

good_parallax['A_G_error'] = 0.15 * good_parallax['A_G']  # rough approximation

good_parallax['abs_mag_error'] = np.sqrt(
    good_parallax['mag_error']**2 +
    good_parallax['parallax_mag_error']**2 +
    good_parallax['A_G_error']**2
)

/var/folders/kj/qbnn_d_10sz_s6gyn25kdtrw0000gp/T/ipykernel_22713/3430789958.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  good_parallax['distance_pc'] = 1000 / good_parallax['parallax']
/var/folders/kj/qbnn_d_10sz_s6gyn25kdtrw0000gp/T/ipykernel_22713/3430789958.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  good_parallax['E_bayestar'] = bayestar(coords, mode='best')
/var/folders/kj/qbnn_d_10sz_s6gyn25kdtrw0000gp/T/ipykernel_22713/3430789958.py:16: SettingWithCopyWarning: 
A value is trying to be

In [165]:
# ok now stars with bad parallax
source_ids = parallax_problem['source_id']  # your list

id_list = ",".join(str(s) for s in source_ids)

query = f"""
SELECT g.source_id, g.ra, g.dec, g.parallax,
       bj.r_med_geo, bj.r_lo_geo, bj.r_hi_geo,
       bj.r_med_photogeo, bj.r_lo_photogeo, bj.r_hi_photogeo
FROM gaiadr3.gaia_source AS g
JOIN external.gaiaedr3_distance AS bj
  ON g.source_id = bj.source_id
WHERE g.source_id IN ({id_list})
"""

job = Gaia.launch_job_async(query)
results = job.get_results().to_pandas()
results['source_id'] = results['source_id'].astype(str)

results['r_unc'] = (results['r_hi_photogeo'] - results['r_lo_photogeo']) / 2 #uncertainty = 1/2 range
parallax_problem = parallax_problem.merge(results[['source_id','r_med_photogeo','r_unc']], on='source_id', how='left')

INFO: Query finished. [astroquery.utils.tap.core]


In [166]:
coords = SkyCoord(
    ra=parallax_problem['ra'].values * u.deg,
    dec=parallax_problem['dec'].values * u.deg,
    distance=parallax_problem['r_med_photogeo'].values * u.pc
)

# BAYESTAR returns reddening on its own scale — need mode='best' or samples
parallax_problem['E_bayestar'] = bayestar(coords, mode='best')

E_BV = 0.884 * parallax_problem['E_bayestar']

parallax_problem['A_G'] =  2.74* E_BV

# extinction-corrected apparent magnitude
parallax_problem['G_corrected'] = parallax_problem['phot_g_mean_mag'] - parallax_problem['A_G']

# now redo the earlier steps with G_corrected instead of phot_g_mean_mag
parallax_problem['abs_mag'] = parallax_problem['G_corrected'] - 5 * np.log10(parallax_problem['r_med_photogeo']) + 5


# --- error propagation ---
# magnitude error from flux error: sigma_mag = (2.5 / ln10) * (flux_error / flux)
parallax_problem['mag_error'] = (2.5 / np.log(10)) * (parallax_problem['phot_g_mean_flux_error'] / parallax_problem['phot_g_mean_flux'])

parallax_problem['distance_mag_error'] = (5 / (parallax_problem['r_med_photogeo'] * np.log(10))) * parallax_problem['r_unc']

parallax_problem['A_G_error'] = 0.15 * parallax_problem['A_G']  # rough approximation

parallax_problem['abs_mag_error'] = np.sqrt(
    parallax_problem['mag_error']**2 +
    parallax_problem['distance_mag_error']**2 +
    parallax_problem['A_G_error']**2
)

In [169]:
# Stack the two lookup tables, giving good_parallax priority when a source_id appears in both
combined = pd.concat([
    parallax_problem[['source_id', 'abs_mag', 'abs_mag_error']],
    good_parallax[['source_id','abs_mag', 'abs_mag_error']]
])

In [170]:
ms = ms.merge(combined, on='source_id', how='left')
ms = ms.merge(sed_results, on='source_id', how='left')

In [172]:
ms['verifiable'] = (~ms['abs_mag'].isna() & ~ms['Teff'].isna()).astype(int)

In [173]:
# group by 'fname' and check if all stars in each group have verifiable == 1
group_check = ms.groupby('fname')['verifiable'].all() if ms['verifiable'].dtype == bool \
    else ms.groupby('fname')['verifiable'].apply(lambda x: (x == 1).all())

n_all_verifiable = group_check.sum()
print(n_all_verifiable)

112


In [174]:
# line endpoints: (Teff1, mag1) = (6800, -2), (Teff2, mag2) = (3400, 7)
teff1, mag1 = 6800, -2
teff2, mag2 = 3400, 7

slope = (mag2 - mag1) / (teff2 - teff1)
intercept = mag1 - slope * teff1

def line_mag(teff):
    return slope * teff + intercept

# "below" the line visually (inverted y-axis) means abs_mag is LARGER than the line value
below_line = ms[ms['abs_mag'] > line_mag(ms['Teff'])].copy()
above_line = ms[ms['abs_mag'] <= line_mag(ms['Teff'])].copy()


In [175]:
ms['rg'] = np.nan
ms.loc[ms['source_id'].isin(below_line['source_id']), 'rg'] = 0
ms.loc[ms['source_id'].isin(above_line['source_id']), 'rg'] = 1

In [176]:
group_check = ms.groupby('fname').apply(
    lambda g: (g['verifiable'] == 1).all() and (g['rg'] == 1).sum() == 1
)

n_cases = group_check.sum()
print(n_cases)

84


/var/folders/kj/qbnn_d_10sz_s6gyn25kdtrw0000gp/T/ipykernel_22713/2160845898.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  group_check = ms.groupby('fname').apply(


In [177]:
g = ms.groupby('fname')
all_verifiable = g['verifiable'].transform(lambda x: (x == 1).all())
rg_count = g['rg'].transform(lambda x: (x == 1).sum())

qualifying_mask = all_verifiable & (rg_count == 1)

# isolate just the rg == 1 stars from those qualifying groups
rg_stars = ms.loc[qualifying_mask & (ms['rg'] == 1)].copy()

In [178]:
#isolate rows where source id has been found in multiple fnames
not_duplicated = rg_stars[~rg_stars.duplicated('source_id',keep=False)]
duplicated = rg_stars[rg_stars.duplicated('source_id',keep=False)].sort_values(by='source_id')

In [179]:
#for a given group, keep stars with the highest 'syd success'
duplicated = duplicated[duplicated['syd_success'] == duplicated.groupby('source_id')['syd_success'].transform('max')]
#for a given group, keep the star with the lowest numax uncertainty
duplicated = duplicated[duplicated['numax_sig'] == duplicated.groupby('source_id')['numax_sig'].transform('min')]

In [180]:
rg_stars = pd.concat([not_duplicated, duplicated]).sort_values(by='source_id')

In [182]:
rg_stars.to_csv('../data/ms_true_sources.csv', index=False)